In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import torch
import numpy as np
import evaluate
from google.colab import drive
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

drive.mount('/content/drive')
output_directory = '/content/drive/MyDrive/finbert_finetuned'

Mounted at /content/drive


In [ ]:
df_extracted_edgar_china_sentences = pd.read_excel('/content/drive/MyDrive/EDGAR/extracted_edgar_china_sentences.xlsx')
display(df_extracted_edgar_china_sentences.head())

,ticker,year,document_type,sentence,filename
0,AMD,2020,10-K,We conduct product and system research and dev...,mda_2020_10-K_0000002488-20-000008.txt.txt
1,AMD,2020,10-K,"For example, the European Union (EU) and China...",mda_2020_10-K_0000002488-20-000008.txt.txt
2,AMD,2020,10-K,"A number of jurisdictions including the EU, Au...",mda_2020_10-K_0000002488-20-000008.txt.txt
3,AMD,2022,10-K,We conduct product and system research and dev...,mda_2022_10-K_0000002488-22-000016.txt.txt
4,AMD,2022,10-K,"For example, the European Union (EU) and China...",mda_2022_10-K_0000002488-22-000016.txt.txt


In [ ]:
df_extracted_earnings_china_sentences = pd.read_excel('/content/drive/MyDrive/EarningsCall/extracted_earnings_china_sentences.xlsx')
display(df_extracted_earnings_china_sentences.head())

,ticker,year,document_type,sentence,filename
0,TER,2015,EarningsCall,"So one of the factors there will be in Asia, a...",20150129_Teradyne_Inc.txt
1,TER,2017,EarningsCall,"On the other end, you have emerging companies,...",20170126_Teradyne_Inc.txt
2,TER,2017,EarningsCall,"And then, I guess just tied to that, also, how...",20170126_Teradyne_Inc.txt
3,TER,2017,EarningsCall,"Obviously, we hear a lot of investment on the ...",20170126_Teradyne_Inc.txt
4,TER,2017,EarningsCall,Just curious how you're positioned in China an...,20170126_Teradyne_Inc.txt


In [ ]:
df_extracted_edgar_earnings_china_sentences = pd.concat([
    df_extracted_edgar_china_sentences,
    df_extracted_earnings_china_sentences
], ignore_index=True)

display(df_extracted_edgar_earnings_china_sentences.head())

,ticker,year,document_type,sentence,filename
0,AMD,2020,10-K,We conduct product and system research and dev...,mda_2020_10-K_0000002488-20-000008.txt.txt
1,AMD,2020,10-K,"For example, the European Union (EU) and China...",mda_2020_10-K_0000002488-20-000008.txt.txt
2,AMD,2020,10-K,"A number of jurisdictions including the EU, Au...",mda_2020_10-K_0000002488-20-000008.txt.txt
3,AMD,2022,10-K,We conduct product and system research and dev...,mda_2022_10-K_0000002488-22-000016.txt.txt
4,AMD,2022,10-K,"For example, the European Union (EU) and China...",mda_2022_10-K_0000002488-22-000016.txt.txt


In [ ]:
model_name = "ProsusAI/finbert"

print("Loading tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    ignore_mismatched_sizes=True
)

Loading tokenizer and model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def get_sentiment_scores(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    outputs = model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    score = probs.detach().numpy()[0]
    predicted_class_id = probs.argmax().item()
    predicted_label = model.config.id2label[predicted_class_id]

    print(f"Text: {text}\nScore: {score}\nLabel: {predicted_label}")
    return predicted_label

In [ ]:
df_extracted_edgar_earnings_china_sentences['score'] = df_extracted_edgar_earnings_china_sentences['sentence'].apply(get_sentiment_scores)
display(df_extracted_edgar_earnings_china_sentences.head())

Streaming output truncated to the last 5000 lines.
Text: And that was very helpful in that environment
that we experienced here in China last year.
Score: [0.8745321  0.00930792 0.11615995]
Label: positive
Text: And similar to China, we also have a very strong execution by our sales and marketing
organization in Europe.
Score: [0.89681274 0.00724187 0.09594535]
Label: positive
Text: I guess first Olivier, based on the feedback you're
hearing from either internally or externally how confident are you at this point that things get
going again next Monday in China?
Score: [0.20637512 0.01338455 0.78024036]
Label: neutral
Text: It's -- even an area that we right now see in our Chinese business.
Score: [0.08269633 0.01359402 0.90370965]
Label: neutral
Text: A - `Shawn P. Vadala, Chief Financial Officer `
Yes, we might see something in field service but that's a smaller part of the Chinese business
versus our global average.
Score: [0.11952638 0.01888832 0.86158526]
Label: neutral
Text: And 

,ticker,year,document_type,sentence,filename,score
0,AMD,2020,10-K,We conduct product and system research and dev...,mda_2020_10-K_0000002488-20-000008.txt.txt,neutral
1,AMD,2020,10-K,"For example, the European Union (EU) and China...",mda_2020_10-K_0000002488-20-000008.txt.txt,neutral
2,AMD,2020,10-K,"A number of jurisdictions including the EU, Au...",mda_2020_10-K_0000002488-20-000008.txt.txt,neutral
3,AMD,2022,10-K,We conduct product and system research and dev...,mda_2022_10-K_0000002488-22-000016.txt.txt,neutral
4,AMD,2022,10-K,"For example, the European Union (EU) and China...",mda_2022_10-K_0000002488-22-000016.txt.txt,neutral


In [ ]:
df_extracted_edgar_earnings_china_sentences.to_excel('extracted_edgar_earnings_china_sentences.xlsx', index=False)
print("File saved successfully.")

File saved successfully.
